# Translation to Croatian
### pre-flight audit

Goal: count exactly how many Unicode code points the EN → HR translation job will send to the Google Cloud Translation API, before any paid call is made.

Google bills text translation per character, where one character equals one Unicode code point. The free trial credit is 300 dollars, the NMT rate is 20 dollars per 1M characters, and the always-free monthly quota is 500,000 characters. The job described in the Stage 4 plan is expected to land near 5.4M characters; this notebook confirms the exact number per dataset, per column, per split, so that no decision downstream is taken on an estimate.

Scope under audit:
- TherapistQA: `patient_question` — train, val, test. Used to train the binary distortion-vs-no-distortion classifier.
- PatternReframe: `thought`, `reframe_1`, `reframe_2`, `reframe_3` — train, val, test. `thought` trains the 10-category classifier; the three `reframe_*` columns are reference reframes for the secondary reframing evaluation in Chapter 5.
- Cognitive Reframing Dataset (Sharma et al., 2023): `situation`, `thought`, `reframe` — single file, 600 rows. Primary reframing evaluation corpus in Chapter 5 because it is the only public dataset with an explicit `situation` column, which matches the production web app's input structure (situation + thought).

Output: a per-column breakdown, a per-split breakdown, a grand total, and the expected cost in USD against the \$300 trial credit.

## 1. Imports and paths

In [1]:
from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("./processed")

# --- TherapistQA ---
TQA_FILES = {
    "train": PROCESSED_DIR / "therapistqa_train.csv",
    "val":   PROCESSED_DIR / "therapistqa_val.csv",
    "test":  PROCESSED_DIR / "therapistqa_test.csv",
}
TQA_COLS = ["patient_question"]

# --- PatternReframe ---
PR_FILES = {
    "train": PROCESSED_DIR / "patternreframe_train.csv",
    "val":   PROCESSED_DIR / "patternreframe_val.csv",
    "test":  PROCESSED_DIR / "patternreframe_test.csv",
}
PR_COLS = ["thought", "reframe_1", "reframe_2", "reframe_3"]

# --- Cognitive Reframing Dataset (single file, no splits) ---
CR_FILES = {
    "all": PROCESSED_DIR / "cognitive_reframing.csv",
}
CR_COLS = ["situation", "thought", "reframe"]

# Google Cloud Translation pricing (NMT model, EN → HR, as of 2026-05).
USD_PER_1M_CHARS = 20.0
FREE_MONTHLY_CHARS = 500_000        # Always-free; resets every calendar month.
TRIAL_CREDIT_USD = 300.0            # 90-day free trial.

## 2. Verify files exist before counting

In [2]:
missing = []
for label, path in {**TQA_FILES, **PR_FILES, **CR_FILES}.items():
    if not path.exists():
        missing.append(str(path))

if missing:
    print("MISSING FILES — fix paths in cell 1 before proceeding:")
    for m in missing:
        print(f"  {m}")
else:
    print("All 7 expected CSVs found.")

All 7 expected CSVs found.


## 3. Verify expected columns are present in each file

In [3]:
def check_columns(path, expected_cols):
    df = pd.read_csv(path, nrows=1)
    return [c for c in expected_cols if c not in df.columns]

all_good = True

for split, path in TQA_FILES.items():
    miss = check_columns(path, TQA_COLS)
    if miss:
        print(f"TQA {split}: MISSING columns {miss}")
        all_good = False
    else:
        print(f"TQA {split}: OK")

for split, path in PR_FILES.items():
    miss = check_columns(path, PR_COLS)
    if miss:
        print(f"PR  {split}: MISSING columns {miss}")
        all_good = False
    else:
        print(f"PR  {split}: OK")

for split, path in CR_FILES.items():
    miss = check_columns(path, CR_COLS)
    if miss:
        print(f"CR  {split}: MISSING columns {miss}")
        all_good = False
    else:
        print(f"CR  {split}: OK")

if all_good:
    print("\nAll required columns present in all files.")

TQA train: OK
TQA val: OK
TQA test: OK
PR  train: OK
PR  val: OK
PR  test: OK
CR  all: OK

All required columns present in all files.


## 4. Character counter

Google counts every code point in the *input* — including whitespace and untranslated tokens — once per call. `len()` in Python 3 is already code-point based, which matches Google's billing exactly. Empty / NaN cells contribute zero characters.

Note on duplicates: the API charges per call, not per unique string. If the same string appears in two rows, it is billed twice unless the script deduplicates before sending. The audit reports both totals so the saving from dedup is visible. For PR reframes in particular, some thoughts have identical or near-identical rewrites across slots — dedup is worth doing in Phase D.

In [5]:
def column_char_stats(path, cols):
    """Return a per-column dict with raw char count, unique char count, n rows, n non-null."""
    df = pd.read_csv(path)
    stats = {}
    for col in cols:
        series = df[col].dropna().astype(str)
        raw_chars = int(series.str.len().sum())
        unique_strs = series.drop_duplicates()
        unique_chars = int(unique_strs.str.len().sum())
        stats[col] = {
            "n_rows": int(len(df)),
            "n_nonnull": int(len(series)),
            "n_unique_strings": int(len(unique_strs)),
            "raw_chars": raw_chars,
            "unique_chars": unique_chars,
            "mean_chars_per_row": raw_chars / max(len(series), 1),
        }
    return stats

## 5. Run the count across all seven files

In [6]:
rows = []

for split, path in TQA_FILES.items():
    stats = column_char_stats(path, TQA_COLS)
    for col, s in stats.items():
        rows.append({"dataset": "TherapistQA", "split": split, "column": col, **s})

for split, path in PR_FILES.items():
    stats = column_char_stats(path, PR_COLS)
    for col, s in stats.items():
        rows.append({"dataset": "PatternReframe", "split": split, "column": col, **s})

for split, path in CR_FILES.items():
    stats = column_char_stats(path, CR_COLS)
    for col, s in stats.items():
        rows.append({"dataset": "CognitiveReframing", "split": split, "column": col, **s})

report = pd.DataFrame(rows)
report = report[["dataset", "split", "column", "n_rows", "n_nonnull",
                 "n_unique_strings", "raw_chars", "unique_chars", "mean_chars_per_row"]]
report

,dataset,split,column,n_rows,n_nonnull,n_unique_strings,raw_chars,unique_chars,mean_chars_per_row
0,TherapistQA,train,patient_question,1768,1768,1768,1527679,1527679,864.071833
1,TherapistQA,val,patient_question,379,379,379,339655,339655,896.187335
2,TherapistQA,test,patient_question,379,379,379,331030,331030,873.430079
3,PatternReframe,train,thought,1920,1920,1920,172969,172969,90.088021
4,PatternReframe,train,reframe_1,1920,1920,1920,203691,203691,106.089062
5,PatternReframe,train,reframe_2,1920,1868,1868,198255,198255,106.132227
6,PatternReframe,train,reframe_3,1920,1461,1461,155608,155608,106.507871
7,PatternReframe,val,thought,961,961,961,86446,86446,89.954214
8,PatternReframe,val,reframe_1,961,961,961,101973,101973,106.111342
9,PatternReframe,val,reframe_2,961,924,924,97437,97437,105.451299


## 6. Per-dataset and grand totals

In [7]:
per_dataset = report.groupby("dataset")[["raw_chars", "unique_chars"]].sum()
per_dataset["raw_M"] = per_dataset["raw_chars"] / 1_000_000
per_dataset["unique_M"] = per_dataset["unique_chars"] / 1_000_000
per_dataset["raw_cost_usd"] = per_dataset["raw_M"] * USD_PER_1M_CHARS
per_dataset["unique_cost_usd"] = per_dataset["unique_M"] * USD_PER_1M_CHARS
per_dataset

,raw_chars,unique_chars,raw_M,unique_M,raw_cost_usd,unique_cost_usd
dataset,,,,,,
CognitiveReframing,151914,109812,0.151914,0.109812,3.03828,2.19624
PatternReframe,3645194,3644848,3.645194,3.644848,72.90388,72.89696
TherapistQA,2198364,2198364,2.198364,2.198364,43.96728,43.96728


In [8]:
grand_raw = int(report["raw_chars"].sum())
grand_unique = int(report["unique_chars"].sum())

raw_cost = grand_raw / 1_000_000 * USD_PER_1M_CHARS
unique_cost = grand_unique / 1_000_000 * USD_PER_1M_CHARS

# After always-free 500k/month (assume single calendar month for worst case).
raw_billable = max(0, grand_raw - FREE_MONTHLY_CHARS)
raw_cost_after_free = raw_billable / 1_000_000 * USD_PER_1M_CHARS

print("=" * 60)
print("GRAND TOTALS — Stage 4 EN → HR translation job")
print("=" * 60)
print(f"Raw characters (every row, every column):   {grand_raw:>12,}  ({grand_raw/1e6:.2f}M)")
print(f"Unique characters (dedup within columns):   {grand_unique:>12,}  ({grand_unique/1e6:.2f}M)")
print()
print(f"Cost at $20/1M, no dedup, no free quota:    ${raw_cost:>10.2f}")
print(f"Cost at $20/1M, with dedup, no free quota:  ${unique_cost:>10.2f}")
print(f"Cost at $20/1M, no dedup, minus 500k free:  ${raw_cost_after_free:>10.2f}")
print()
print(f"$300 trial credit remaining after job:      ${TRIAL_CREDIT_USD - raw_cost_after_free:>10.2f}")
print()
if raw_cost_after_free < TRIAL_CREDIT_USD:
    print("✓ Full job fits inside the free trial credit.")
else:
    print("✗ Full job EXCEEDS the free trial credit — reconsider scope.")

GRAND TOTALS — Stage 4 EN → HR translation job
Raw characters (every row, every column):      5,995,472  (6.00M)
Unique characters (dedup within columns):      5,953,024  (5.95M)

Cost at $20/1M, no dedup, no free quota:    $    119.91
Cost at $20/1M, with dedup, no free quota:  $    119.06
Cost at $20/1M, no dedup, minus 500k free:  $    109.91

$300 trial credit remaining after job:      $    190.09

✓ Full job fits inside the free trial credit.


## 7. Save audit report to disk

The CSV gets cited in Chapter 3.4 (Data translation into Croatian) and Section 4.4 (Multilingual models). It records the exact volume the experiment was run against.

In [9]:
AUDIT_DIR = Path("processed") / "translation_audit"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

report.to_csv(AUDIT_DIR / "preflight_char_count.csv", index=False)
per_dataset.to_csv(AUDIT_DIR / "preflight_per_dataset_totals.csv")

print(f"Audit report written to {AUDIT_DIR.resolve()}")

Audit report written to /Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/processed/translation_audit
